In [65]:
import numpy as np
import pandas as pd

import json
from scipy.io import arff

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [66]:
def resample_data_set(inputs_, outputs_, params):
    sample_size = inputs_.shape[0] - 1
    indices = np.arange(sample_size + 1)
    np.random.shuffle(indices)

    inputs_ = inputs_[indices, :]
    outputs_ = outputs_[indices, :]

    inputs, outputs = (inputs_[:-1, :], outputs_[:-1, :])
    input_test, output_test = (
        inputs_[-1, :].reshape(1, -1),
        outputs_[-1, :].reshape(1, -1),
    )

    inputs_train, inputs_calibration, outputs_train, outputs_calibration = (
        train_test_split(inputs, outputs, test_size=params["cal_size"])
    )

    input_scaler = StandardScaler()
    scaled_inputs_train = input_scaler.fit_transform(inputs_train)
    scaled_inputs_calibration = input_scaler.transform(inputs_calibration)
    scaled_input_test = input_scaler.transform(input_test)

    (
        scaled_inputs_selection,
        scaled_inputs_proper_cal,
        outputs_selection,
        outputs_proper_cal,
    ) = train_test_split(
        scaled_inputs_calibration,
        outputs_calibration,
        test_size=params["proper_cal_size"],
    )

    return (
        (scaled_inputs_train, outputs_train),
        (scaled_inputs_calibration, outputs_calibration),
        (
            scaled_inputs_selection,
            scaled_inputs_proper_cal,
            outputs_selection,
            outputs_proper_cal,
        ),
        (scaled_input_test, output_test),
    )

In [67]:
def reader(path_params):
    with open(path_params, "r") as file:
        params = json.load(file)
    print(params)
    return params

In [68]:
arff_file = arff.loadarff("../datasets/yeast.arff")
data_frame = pd.DataFrame(arff_file[0])
inputs_ = (data_frame[data_frame.columns[:103]]).to_numpy()
outputs_ = np.int64((data_frame[data_frame.columns[103:]]).to_numpy() == b"TRUE")

In [69]:
params_global = reader("params/yeast.json")

{'data': {'cal_size': 0.5, 'proper_cal_size': 0.5}}


In [71]:
(
    (scaled_inputs_train, outputs_train),
    (scaled_inputs_calibration, outputs_calibration),
    (
        scaled_inputs_selection,
        scaled_inputs_proper_cal,
        outputs_selection,
        outputs_proper_cal,
    ),
    (scaled_input_test, output_test),
) = resample_data_set(inputs_, outputs_, params_global["data"])

In [72]:
print(scaled_inputs_train.shape)
print(outputs_train.shape)

print(scaled_inputs_calibration.shape)
print(outputs_calibration.shape)

print(scaled_inputs_selection.shape)
print(outputs_selection.shape)

print(scaled_inputs_proper_cal.shape)
print(outputs_proper_cal.shape)

print(scaled_input_test.shape)
print(output_test.shape)

(1208, 103)
(1208, 14)
(1208, 103)
(1208, 14)
(604, 103)
(604, 14)
(604, 103)
(604, 14)
(1, 103)
(1, 14)


In [81]:
print("The average number of label is {}.".format(outputs_.sum(axis=1).mean()))

The average number of label is 4.237070748862226.
